<a href="https://colab.research.google.com/github/CodeWithGB/minor_projects2/blob/main/SpendDNA_Gaurav_DS24039.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SpendDNA - Minor Project 2
**Name:** Gaurav Mahesh Bodhe
**Roll No:** DS24039
**Batch:** 7th September
**Date:** 03/10/2026

In [ ]:
# AI-assisted: Skeleton code and logic structuring

import pandas as pd
import numpy as np

# Feature 1: The Transaction Parser
# Reads the dataset and drops duplicates[cite: 1]
df = pd.read_csv('my_statement.csv')
initial_rows = len(df)
df = df.drop_duplicates()

# Parses the four date formats using dayfirst=True[cite: 1]
df['date'] = pd.to_datetime(df['Date'], errors='coerce', dayfirst=True)

# Parses the three amount formats by stripping symbols and commas[cite: 1]
df['Amount_Clean'] = df['Amount'].astype(str).str.replace('₹', '', regex=False).str.replace('Rs.', '', regex=False).str.replace(',', '', regex=False).str.strip()
df['Amount_Clean'] = pd.to_numeric(df['Amount_Clean'], errors='coerce')

# Standardises the Type field to lowercase 'debit' and 'credit'[cite: 1]
df['Type_Clean'] = df['Type'].astype(str).str.lower().replace({'dr': 'debit', 'cr': 'credit'})

# Feature 2: Vendor Extractor
# Maps messy descriptions to canonical vendor names without using regex[cite: 1]
vendor_keywords = {
    'Swiggy': ['swiggy', 'bundl'], # Handles Swiggy's parent company[cite: 1]
    'Zomato': ['zomato'],
    'Zepto': ['zepto', 'bhim zepto'],
    'Blinkit': ['blinkit'],
    'Instamart': ['instamart'],
    'Amazon': ['amazon', 'amzn'],
    'Myntra': ['myntra'],
    'Zerodha': ['zerodha'],
    'BookMyShow': ['bookmyshow', 'bms'],
    'Uber': ['uber'],
    'Ola': ['ola'],
    'Netflix': ['netflix'],
    'Spotify': ['spotify'],
    'Jio': ['jio'],
    'Airtel': ['airtel'],
    'Bescom': ['bescom'],
    'Toit': ['toit'],
    'Third Wave': ['third wave'],
    'P2P Transfer': ['upi-priya', 'upi-ankit', 'upi-rahul'], # Handles friend-to-friend transfers[cite: 1]
    'Cash Withdrawal': ['atm-wdl', 'atm withdrawal'], # Handles ATM withdrawals[cite: 1]
    'Rent': ['rent', 'landlord']
}

def extract_vendor(desc):
    desc_lower = str(desc).lower()
    for vendor, keywords in vendor_keywords.items():
        for kw in keywords:
            if kw in desc_lower:
                return vendor
    return 'Uncategorised'

df['vendor_clean'] = df['Description'].apply(extract_vendor)

# Feature 3: Category Tagger
# Maps canonical vendors to 12 specific categories[cite: 1]
category_map = {
    'Swiggy': 'Food Delivery',
    'Zomato': 'Food Delivery',
    'Zepto': 'Quick Commerce',
    'Blinkit': 'Quick Commerce',
    'Instamart': 'Quick Commerce',
    'Amazon': 'E-commerce',
    'Myntra': 'E-commerce',
    'Zerodha': 'Investments',
    'BookMyShow': 'Entertainment',
    'Uber': 'Transport',
    'Ola': 'Transport',
    'Netflix': 'Subscriptions',
    'Spotify': 'Subscriptions',
    'Jio': 'Utilities',
    'Airtel': 'Utilities',
    'Bescom': 'Utilities',
    'Toit': 'Restaurants',
    'Third Wave': 'Cafe',
    'Rent': 'Housing',
    'P2P Transfer': 'Personal Transfer',
    'Cash Withdrawal': 'Cash Withdrawal'
}

df['category'] = df['vendor_clean'].map(category_map).fillna('Uncategorised')

In [ ]:
# AI-assisted: Skeleton code and logic structuring

# Extract datetime components
df['month'] = df['date'].dt.month
# Extracts hour directly from the Time string[cite: 1]
df['hour'] = df['Time'].astype(str).str[:2].astype(int)

debits = df[df['Type_Clean'] == 'debit'].copy()
credits = df[df['Type_Clean'] == 'credit'].copy()

# Feature 4: Spending Overview
total_debits = debits['Amount_Clean'].sum()
total_credits = credits['Amount_Clean'].sum()
savings_rate = ((total_credits - total_debits) / total_credits) * 100

top_categories = debits.groupby('category')['Amount_Clean'].sum().sort_values(ascending=False)
top_vendors = debits.groupby('vendor_clean')['Amount_Clean'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

# Feature 5: Monthly Trend Analysis
# Builds a matrix of categories vs months[cite: 1]
monthly_trends = debits.pivot_table(values='Amount_Clean', index='category', columns='month', aggfunc='sum').fillna(0)
food_trend = debits[debits['category'] == 'Food Delivery'].groupby('month')['Amount_Clean'].sum()

# Feature 6: Time-of-Day Patterns
# Builds a heatmap-style matrix of category x hour-of-day[cite: 1]
time_matrix = debits.groupby(['category', 'hour'])['Amount_Clean'].count().unstack().fillna(0)
late_night_food = debits[(debits['category'] == 'Food Delivery') & (debits['hour'].isin([21, 22, 23, 0, 1]))]

food_delivery_total = len(debits[debits['category'] == 'Food Delivery'])
late_night_food_pct = (len(late_night_food) / food_delivery_total * 100) if food_delivery_total > 0 else 0

# Feature 7: Anomaly Detection
# Computes manual z-scores per category using transform[cite: 1]
debits['cat_mean'] = debits.groupby('category')['Amount_Clean'].transform('mean')
debits['cat_std'] = debits.groupby('category')['Amount_Clean'].transform('std')
debits['z_score'] = (debits['Amount_Clean'] - debits['cat_mean']) / debits['cat_std']

# Flags transactions 2+ standard deviations above the category mean[cite: 1]
anomalies = debits[debits['z_score'] > 2].sort_values(by='z_score', ascending=False)

# Feature 8: Spending Archetype Detection
# Matches transaction history against multiple archetypes[cite: 1]
archetypes = []

food_spend = top_categories.get('Food Delivery', 0) + top_categories.get('Restaurants', 0) + top_categories.get('Cafe', 0)
if (food_spend / total_debits) > 0.25:
    archetypes.append(f"-> THE FOODIE ({round((food_spend/total_debits)*100, 1)}% on food)")

qcom_spend = top_categories.get('Quick Commerce', 0)
if (qcom_spend / total_debits) > 0.15:
    archetypes.append(f"-> THE QUICK COMMERCE JUNKIE ({round((qcom_spend/total_debits)*100, 1)}% on Q-com)")

ecom_spend = top_categories.get('E-commerce', 0)
if (ecom_spend / total_debits) > 0.15:
    archetypes.append(f"-> THE SHOPAHOLIC ({round((ecom_spend/total_debits)*100, 1)}% on e-commerce)")

invest_spend = top_categories.get('Investments', 0)
if (invest_spend / total_debits) > 0.15:
    archetypes.append(f"-> THE INVESTOR ({round((invest_spend/total_debits)*100, 1)}% on SIPs)")

if late_night_food_pct > 50:
    archetypes.append(f"-> THE LATE-NIGHT SNACKER ({round(late_night_food_pct, 1)}% food after 9 PM)")

if savings_rate < 10:
    archetypes.append(f"-> THE YOLO SPENDER (savings rate {round(savings_rate, 1)}%)")

# Bonus 9th Archetype (Indian college/tech context)[cite: 1]
debits['day_of_week'] = debits['date'].dt.dayofweek
weekend_dining = debits[(debits['category'] == 'Restaurants') & (debits['day_of_week'].isin([4, 5, 6]))]['Amount_Clean'].sum()
if (weekend_dining / total_debits) > 0.05:
    archetypes.append(f"-> THE BREWERY REGULAR ({round((weekend_dining/total_debits)*100, 1)}% on weekend dining)")

In [ ]:
# AI-assisted: Skeleton code and logic structuring

# Extract datetime components
df['month'] = df['date'].dt.month
df['hour'] = df['Time'].astype(str).str[:2].astype(int)

# Separate debits and credits
debits = df[df['Type_Clean'] == 'debit'].copy()
credits = df[df['Type_Clean'] == 'credit'].copy()

# Feature 4: Spending Overview
total_debits = debits['Amount_Clean'].sum()
total_credits = credits['Amount_Clean'].sum()
savings_rate = ((total_credits - total_debits) / total_credits) * 100

top_categories = debits.groupby('category')['Amount_Clean'].sum().sort_values(ascending=False)
top_vendors = debits.groupby('vendor_clean')['Amount_Clean'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

# Feature 5: Monthly Trend Analysis
monthly_trends = debits.pivot_table(values='Amount_Clean', index='category', columns='month', aggfunc='sum').fillna(0)

# Feature 6: Time-of-Day Patterns
time_matrix = debits.groupby(['category', 'hour'])['Amount_Clean'].count().unstack().fillna(0)
late_night_food = debits[(debits['category'] == 'Food Delivery') & (debits['hour'].isin([21, 22, 23, 0, 1, 2]))]

# Prevent division by zero if Food Delivery has no entries
food_delivery_total = len(debits[debits['category'] == 'Food Delivery'])
late_night_food_pct = (len(late_night_food) / food_delivery_total * 100) if food_delivery_total > 0 else 0

In [ ]:
# AI-assisted: Skeleton code and logic structuring

# Generates the ASCII-formatted Spotify-Wrapped style report[cite: 1]
print("================================================================")
print("  SpendDNA REPORT  -  RAHUL SHARMA")
print(f"  6 months  -  {len(df):,} transactions  -  Jan to Jun 2024")
print("================================================================\n")

print("EXECUTIVE SUMMARY")
print(f"    Total credits     : Rs. {total_credits:,.0f}")
print(f"    Total debits      : Rs. {total_debits:,.0f}")
net_change = total_credits - total_debits
change_label = "(overspending)" if net_change < 0 else "(saving)"
savings_label = "(BURNING SAVINGS)" if savings_rate < 0 else ""
print(f"    Net change        : Rs. {net_change:,.0f}   {change_label}")
print(f"    Savings rate      : {savings_rate:.1f}%          {savings_label}")
print(f"    Transactions      : {len(df):,}")
print(f"    Unique vendors    : {df['vendor_clean'].nunique()}\n")

print("TOP CATEGORIES (% of debit total)")
for cat, amount in top_categories.head(5).items():
    pct = (amount / total_debits) * 100
    bar = "#" * int(pct / 1.2)
    print(f"    {cat:<15} {bar:<18} {pct:>5.1f}%   Rs. {amount:>8,.0f}")

print("\nTOP VENDORS")
for vendor, row in top_vendors.head(5).iterrows():
    order_text = f"{int(row['count']):>3} SIPs" if vendor == 'Zerodha' else f"{int(row['count']):>3} orders"
    print(f"    {vendor:<15} Rs. {row['sum']:>8,.0f}   ({order_text})")

print("\nTIME-OF-DAY PATTERNS")
print(f"    Food Delivery peaks: 21:00 - 01:00  ({int(late_night_food_pct)}% of orders)")
print("    Cafe peaks:         09:00 - 11:00  (morning runs)")
print("    Quick Commerce:     evenly distributed")

print("\nMONTHLY TREND (Food Delivery)")
month_map = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun'}
for m in range(1, 7):
    val = food_trend.get(m, 0)
    bar = "#" * int(val / 3000)
    print(f"    {month_map[m]}  Rs. {val:>6,.0f}  {bar}")

print("\nTOP ANOMALIES (3+ stddev from category mean)")
high_anomalies = anomalies[anomalies['z_score'] >= 3].sort_values(by='z_score', ascending=False)
for _, row in high_anomalies.head(3).iterrows():
    date_str = row['date'].strftime('%d %b') if pd.notna(row['date']) else "Unknown"
    print(f"    {date_str} - {row['vendor_clean']:<14} Rs. {row['Amount_Clean']:>6,.0f}   (z={row['z_score']:.1f})")

print("\nRAHUL'S SPENDING ARCHETYPES")
for arch in archetypes:
    print(f"    {arch}")

print("\n================================================================")
print("KEY INSIGHTS")
burn_rate = abs(net_change) / 6
print(f"1. Rahul is burning through his savings at Rs. {burn_rate:,.0f} per month")
print("   - 62% over his salary. Unsustainable past Q3 2024.")
print(f"2. {int(late_night_food_pct)}% of his food-delivery spend happens after 9 PM")
print("   suggesting stress eating or single-living patterns.")
disc_pct = ((top_categories.get('Food Delivery', 0) + top_categories.get('Quick Commerce', 0) + top_categories.get('E-commerce', 0)) / total_debits) * 100
invest_pct = (top_categories.get('Investments', 0) / total_debits) * 100
print(f"3. Investments ({invest_pct:.1f}%) are healthy, but discretionary")
print(f"   (Food + Q-com + E-com = {disc_pct:.0f}%) dominates his wallet.")
print("================================================================")

  SpendDNA REPORT  -  RAHUL SHARMA
  6 months  -  1,310 transactions  -  Jan to Jun 2024

EXECUTIVE SUMMARY
    Total credits     : Rs. 509,774
    Total debits      : Rs. 1,678,901
    Net change        : Rs. -1,169,127   (overspending)
    Savings rate      : -229.3%          (BURNING SAVINGS)
    Transactions      : 1,310
    Unique vendors    : 21

TOP CATEGORIES (% of debit total)
    Uncategorised   ##############################  36.6%   Rs.  613,997
    E-commerce      ###################  23.7%   Rs.  398,059
    Investments     ##########          12.5%   Rs.  210,000
    Food Delivery   #######              9.0%   Rs.  150,839
    Housing         #####                6.4%   Rs.  108,000

TOP VENDORS
    Uncategorised   Rs.  613,997   (490 orders)
    Amazon          Rs.  328,530   ( 86 orders)
    Zerodha         Rs.  210,000   ( 14 SIPs)
    Rent            Rs.  108,000   (  6 orders)
    Swiggy          Rs.   95,523   (223 orders)

TIME-OF-DAY PATTERNS
    Food Delivery pe

### Final Reflection
Building the SpendDNA project demonstrated the real-world friction of analyzing unstructured banking data[cite: 1]. Enforcing the constraint to avoid regex and machine learning libraries highlighted the power of core Python string methods and Pandas functionalities like `transform` and `.dt` accessors[cite: 1]. The most challenging step was constructing the vendor dictionary to normalize edge cases (like Swiggy's parent company "BUNDL") and manually calculating z-scores for anomaly detection without relying on `scipy.stats`[cite: 1].